# RV recovery step diagnostics (LFC wavesol)

<span style="color:red">The HISPEC DRP is in active development. Please submit issues and suggestions on the [GitHub](https://github.com/oirlab/HISPEC_DRP).</span>

Same densified BT-Settl lte040 recovery grid as the baseline step-diagnostics
notebook (0, 10, 20, 30, 40, 50 km/s; all finite lines kept with
`outlier_sigma=1e30`). Run the preamble and section 1 first, then sections
2–7 in order. The rate-map and UTR paths use two simulated calibrations so
recovery stays free of the ~15–20 m/s continuum/CCF bias:

1. **Truth 1D** (section 1): densify the rest-frame template, Doppler-shift it,
   and verify recovered RV ≈ 0 at injected RV = 0.
2. **LFC WAVE** (section 2): `derive_lfc_wavesol` paints a simulated LFC,
   extracts it like science, and fits an extract-matched wavelength solution.
3. **Identity fiberflat** (section 3): `prepare_stage1_cals(..., fiberflat_mode="identity")`
   caches a ones fiberflat under `OUT/cals/` so Stage-2 blaze is a no-op.
4. **Native L2** (section 4): paint the shifted 1D onto the echelle WAVE grid.
5. **Rate map → Stage 1** (section 5): native rate map, LFC WAVE, identity
   fiberflat, box extract, blaze, recover.
6. **UTR → Stages 0–2** (section 6): same calibrations after a noiseless ramp /
   Stage 0.
7. **Compare** (section 7): which step adds the most bias and RV_ERR.

A paint-matched continuum fiberflat cancels PRF blaze shape but still reweights
the CCF by ~17 m/s on this grid. Keep `FIBERFLAT_MODE = "identity"` for
bias-free simulations; set `"paint_matched"` only to study blaze cancel itself.


In [ ]:
from pathlib import Path

import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

OUT = Path("/path/to/rv_recovery_step_diagnostics_lfc_wavesol")
OUT.mkdir(parents=True, exist_ok=True)
print("output:", OUT.resolve())


## 1. Truth 1D spectrum and RV=0 check


In [ ]:
from hispecdrp.simulations.rv_step_diagnostics import (
    STEP_BULK_RV_KM_S,
    STELLAR_RV_NO_REJECT_KW,
    doppler_shift_spectrum,
    load_densified_template,
    run_step_grid,
    verify_zero_rv_recovery,
)
from hispecdrp.utils.line_mask_utils import (
    paired_line_mask_path_for_template,
    read_mask_generation_meta,
)

FIBER = 1
TEMPLATE = "lte040.0-4.5-0.0a+0.0.BT-Settl.spec.7.h5"
MASK_PATH = paired_line_mask_path_for_template(TEMPLATE, mask_style="ccfopt.delta")
BULK_RV_KM_S = STEP_BULK_RV_KM_S.copy()
meta = read_mask_generation_meta(MASK_PATH) or {}
print(f"mask: {MASK_PATH}")
print(
    f"n_lines={meta.get('n_lines')}  outlier_sigma={STELLAR_RV_NO_REJECT_KW['outlier_sigma']:.0e}"
)
print(f"grid={list(BULK_RV_KM_S)} km/s")

wl_rest, flux_rest = load_densified_template(TEMPLATE)
print(f"truth 1D: n={wl_rest.size}  lambda={wl_rest[0]:.4f}-{wl_rest[-1]:.4f}")

verify_zero_rv_recovery(
    template=TEMPLATE,
    mask_path=str(MASK_PATH),
    fiber=FIBER,
)

TRUTH = {
    float(rv): doppler_shift_spectrum(wl_rest, flux_rest, float(rv))
    for rv in BULK_RV_KM_S
}
summaries = {}


## 2. Derive WAVE from a simulated LFC

Paint an LFC rate map, box-extract like science, and fit `lfc_wavesol`. The product is reused for §§5–6.


In [ ]:
from hispecdrp.simulations.rv_step_diagnostics import derive_lfc_wavesol

lfc_wavesol, lfc_info = derive_lfc_wavesol(
    spectrograph="BSPEC",
    fiber=FIBER,
    output_dir=OUT / "lfc_wavesol_cals",
)
print(lfc_info)
print(
    f"LFC WAVE shape={getattr(lfc_wavesol, f'wave_fib{FIBER}').shape}  "
    f"path={lfc_info['wavesol_path']}"
)


## 3. Simulate the identity fiberflat

Build Stage-1 masters once and replace the continuum fiberflat with an **identity** (ones) product shaped like the LFC WAVE. Blaze correction then divides by `nanmax(ones)=1`, so continuum shape is unchanged and the ~17 m/s CCF reweight goes away.

`FIBERFLAT_MODE` is passed explicitly into the rate-map / UTR recoveries below. Set it to `"paint_matched"` only if you intentionally want the continuum-flattening bias for comparison.


In [ ]:
import numpy as np

from hispecdrp.simulations.ideal_spectrum_rv import (
    FIBERFLAT_IDENTITY,
    prepare_stage1_cals,
)
from hispecdrp.simulations.simulate_rate_map import SimulateRateMap
from hispecdrp.simulations.spectra import quiet_pipeline_logging

FIBERFLAT_MODE = FIBERFLAT_IDENTITY  # "identity" — bias-free for this diagnostic grid

# Seed rate map (in-memory) to size traces / WAVE and write the fiberflat under OUT/cals/.
_seed_rv = 0.0 if 0.0 in TRUTH else float(next(iter(TRUTH)))
with quiet_pipeline_logging():
    srm = SimulateRateMap()
    srm.set_fiber_spectrum(
        fiber_number=FIBER,
        spectrum_wavelengths=TRUTH[_seed_rv][0],
        spectrum_flux=TRUTH[_seed_rv][1],
        continuous=False,
    )
    rate0 = np.asarray(
        srm.create_native_rate_map(spectrograph="BSPEC", save=False),
        dtype=np.float64,
    )
    stage1_cals = prepare_stage1_cals(
        OUT,
        rate0,
        spectrograph="BSPEC",
        fiber=FIBER,
        wavesol=lfc_wavesol,
        fiberflat_mode=FIBERFLAT_MODE,
    )

ff = stage1_cals["fiberflat"]
ff_spec = np.asarray(getattr(ff, f"spec_fib{FIBER}"))
print(f"fiberflat_source={stage1_cals['fiberflat_source']!r}  mode={FIBERFLAT_MODE!r}")
print(f"fiberflat shape={ff_spec.shape}  unique={np.unique(ff_spec)}")
print(f"cached under: {OUT / 'cals'}")
assert stage1_cals["fiberflat_source"] == "identity_ones"
assert np.allclose(ff_spec, 1.0)


## 4. Paint truth onto L2


In [ ]:
from hispecdrp.simulations.rv_step_diagnostics import recover_rv_from_truth_l2


def recover_l2(bulk):
    wl, flux = TRUTH[float(bulk)]
    return recover_rv_from_truth_l2(
        bulk,
        wavelengths=wl,
        flux=flux,
        template=TEMPLATE,
        mask_path=str(MASK_PATH),
        fiber=FIBER,
    )


summaries["L2"] = run_step_grid(
    recover_l2,
    name="L2",
    title="BT-Settl lte040 (truth 1D -> L2)",
    out_path=OUT / "lte040_step_l2.png",
    bulk_rv_km_s=BULK_RV_KM_S,
)


## 5. Rate map -> extract -> blaze (LFC WAVE + identity fiberflat)

Native rate map → box extract → LFC WAVE → blaze with `FIBERFLAT_MODE`. Expect mean bias ≪ 1 m/s, matching §4.


In [ ]:
from hispecdrp.simulations.rv_step_diagnostics import recover_rv_from_truth_rate_map

summaries["rate_map_lfc_wavesol"] = run_step_grid(
    lambda rv: recover_rv_from_truth_rate_map(
        rv,
        wavelengths=TRUTH[float(rv)][0],
        flux=TRUTH[float(rv)][1],
        template=TEMPLATE,
        mask_path=str(MASK_PATH),
        output_dir=OUT / "rate_map",
        fiber=FIBER,
        wavesol=lfc_wavesol,
        fiberflat_mode=FIBERFLAT_MODE,
    ),
    name="rate_map_lfc_wavesol",
    title=f"Rate map -> extract -> blaze (LFC WAVE + {FIBERFLAT_MODE} FF)",
    out_path=OUT / "rate_map_lfc_wavesol_recovery.png",
    bulk_rv_km_s=BULK_RV_KM_S,
)


## 6. UTR -> Stages 0-2 (LFC WAVE + identity fiberflat)

Same LFC WAVE and identity fiberflat as §5, after a noiseless up-the-ramp / Stage-0 path.


In [ ]:
from hispecdrp.simulations.rv_step_diagnostics import recover_rv_from_truth_ramp

summaries["ramp_lfc_wavesol"] = run_step_grid(
    lambda rv: recover_rv_from_truth_ramp(
        rv,
        wavelengths=TRUTH[float(rv)][0],
        flux=TRUTH[float(rv)][1],
        template=TEMPLATE,
        mask_path=str(MASK_PATH),
        output_dir=OUT / "ramp",
        fiber=FIBER,
        wavesol=lfc_wavesol,
        fiberflat_mode=FIBERFLAT_MODE,
    ),
    name="ramp_lfc_wavesol",
    title=f"UTR -> Stages 0-2 (LFC WAVE + {FIBERFLAT_MODE} FF)",
    out_path=OUT / "ramp_lfc_wavesol_recovery.png",
    bulk_rv_km_s=BULK_RV_KM_S,
)


## 7. Compare steps


In [ ]:
from hispecdrp.simulations.plotting import plot_rv_step_comparison
from hispecdrp.simulations.rv_step_diagnostics import compare_step_summaries

order = ["L2", "rate_map_lfc_wavesol", "ramp_lfc_wavesol"]
present = [k for k in order if k in summaries]
compare_step_summaries({k: summaries[k] for k in present})
plot_rv_step_comparison(
    {k: summaries[k] for k in present},
    out_path=OUT / "rv_step_comparison_lfc_wavesol.png",
)
